In [ ]:
import pandas as pd
import numpy as np
import sys
import os
from pathlib import Path

# Define the Project Root
## project root is two levels above notebooks/raw_data_processing
PROJECT_ROOT = Path.cwd().parent.parent
sys.path.append(str(PROJECT_ROOT))

# Import functions
from src.data.census_raw_processing import _filter_census_fsa
from src.data.census_raw_processing import _clean_census_column_names
from src.data.census_raw_processing import _pivot_census_long_to_wide

from src.data.census_raw_processing import build_census_fsa_matrix

# Get file location stuff
from config.paths import CENSUS_RAW
from config.paths import CENSUS_FSA

# Define Input and Output
census_csv_out_filepath = CENSUS_FSA
# Create directory if it doesn't exist
census_csv_out_filepath.parent.mkdir(parents=True, exist_ok=True)
census_filepath = CENSUS_RAW

# achtung! census data is stored locally (not in repo) b/c of size limits! 
# config/paths.py for information... and the readme
# Locally OVERRIDE the config path:
LOCAL_CENSUS_RAW = Path(
    r"C:\Users\john\data\radon_raw\census_canada_2016\98-401-X2016046_eng_CSV\98-401-X2016046_English_CSV_data.csv"
)
census_filepath = LOCAL_CENSUS_RAW


# START HANDLING DATA

# note to self: this could all probably be packaged in build_census_feature_matrix... 
# Specify how to read census data
use_cols = [
    "GEO_CODE (POR)",
    "GEO_LEVEL",
    "Member ID: Profile of Forward Sortation Areas (2247)",
    "Dim: Sex (3): Member ID: [1]: Total - Sex"
]

raw_census_df = pd.read_csv(
    census_filepath,
    encoding="utf-8-sig",
    usecols=use_cols,
    low_memory=False
)

raw_census_df.columns.tolist()
print(raw_census_df["GEO_LEVEL"].unique())
print("Total FSAs:", raw_census_df['GEO_CODE (POR)'].nunique())

print(raw_census_df["GEO_LEVEL"].unique())

df = _clean_census_column_names(raw_census_df)
df = _filter_census_fsa(df)


df_wide = _pivot_census_long_to_wide(df)



[0 2]
Total FSAs: 1642
[0 2]


In [2]:
census_fsa_data = build_census_fsa_matrix(raw_census_df)

print(census_fsa_data.shape)
print(census_fsa_data.columns)

(1641, 29)
Index(['fsa', 'hous_frac_type_single_detached', 'hous_frac_type_highrise',
       'hous_frac_type_other_attached', 'hous_frac_type_movable',
       'hous_avg_rooms', 'hous_frac_major_repair', 'hous_frac_age_pre_1980',
       'hous_frac_age_1981_2000', 'hous_frac_age_post_2001',
       'hous_median_value', 'demogr_pop_2016', 'demogr_num_total_dwellings',
       'demogr_num_occ_dwellings', 'demogr_median_age',
       'demogr_avg_household_size', 'demogr_frac_tenure_owned',
       'demogr_frac_tenure_rented', 'demogr_frac_tenure_band',
       'socioeco_frac_low_income', 'socioeco_median_income',
       'socioeco_frac_high_income', 'socioeco_frac_govt_transfers',
       'socioeco_frac_unemployment_rate', 'socioeco_frac_nonlaborer',
       'socioeco_frac_bachelor_plus', 'socioeco_frac_overcrowded',
       'socioeco_frac_unsuitable_housing', 'socioeco_frac_housing_burden'],
      dtype='str')


In [3]:
census_fsa_data.to_csv(census_csv_out_filepath, index=False)

print("Saved to:", census_csv_out_filepath.resolve())

Saved to: C:\Users\john\Desktop\spring-2026-radon-risk-mapping\data\processed\fsa_census.csv
